# 任意実践：Kaggle Titanicへ提出する

**今日の問い：模擬コンペで覚えた手順を、実際のKaggle過去コンペで再現できるか。**

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

`演習`は全員、`変更して確認`は値を1つ変える練習、`自由課題（任意）`は余裕がある人向けです。
`発展（任意）`・`追加演習`は経験者や自習向けの発展で、飛ばしても本編は完結します。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## 事前準備

1. Kaggleの`Titanic - Machine Learning from Disaster`を開く
2. `Join Competition`からルールへ同意する
3. `Data`画面からデータをダウンロードする
4. ZIP内の`train.csv`、`test.csv`、`gender_submission.csv`を次へ置く

```text
data/kaggle/titanic/
```

このフォルダはGit管理対象外です。会社のデータや認証情報を置かないでください。


In [ ]:
import pandas as pd

titanic_dir = DATA / "kaggle" / "titanic"
train_path = titanic_dir / "train.csv"
test_path = titanic_dir / "test.csv"
ready = train_path.exists() and test_path.exists()
if not ready:
    print("Kaggleからtrain.csvとtest.csvをダウンロードし、次へ置いてください:")
    print(titanic_dir)
else:
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    print("train:", train.shape, "test:", test.shape)
    display(train.head(3))


## ベースラインを検証する

評価指標はaccuracyです。`Survived`を目的変数にし、提出に存在する列だけを使います。


In [ ]:
if ready:
    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score

    target = "Survived"
    features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
    numeric = ["Age", "SibSp", "Parch", "Fare"]
    categorical = ["Pclass", "Sex", "Embarked"]
    preprocess = ColumnTransformer([
        ("数値", SimpleImputer(strategy="median"), numeric),
        ("カテゴリ", Pipeline([
            ("補完", SimpleImputer(strategy="most_frequent")),
            ("one_hot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
    ])
    model = Pipeline([
        ("前処理", preprocess),
        ("モデル", RandomForestClassifier(n_estimators=250, max_depth=5, random_state=42)),
    ])
    X_train, X_valid, y_train, y_valid = train_test_split(
        train[features], train[target], test_size=0.25, random_state=42, stratify=train[target]
    )
    model.fit(X_train, y_train)
    print("ローカル検証accuracy:", round(accuracy_score(y_valid, model.predict(X_valid)), 3))
else:
    print("データ準備後に、このセルをもう一度実行します。")


## 提出CSVを作る

列名と行数を機械的に検査してから、Kaggleの`Submit Predictions`へアップロードします。


In [ ]:
if ready:
    model.fit(train[features], train[target])
    submission = pd.DataFrame({
        "PassengerId": test["PassengerId"],
        "Survived": model.predict(test[features]),
    })
    assert list(submission.columns) == ["PassengerId", "Survived"]
    assert len(submission) == len(test)
    assert submission["PassengerId"].is_unique
    output = ROOT / "workspace" / "titanic_submission.csv"
    submission.to_csv(output, index=False)
    print("提出ファイル:", output)
    display(submission.head())
else:
    print("データ準備後に、このセルをもう一度実行します。")


## 提出後

Leaderboardの点数だけで良し悪しを決めず、ローカル検証、変更点、結果を実験ログへ残します。Kaggle上の他者Notebookは、自分のベースラインを提出した後に読みます。
